# 03 — Optimize: DSPy Pipeline Optimization

Optimize the `HiligaynonChatbot` using DSPy's BootstrapFewShot.

Steps:
1. Evaluate baseline (unoptimized) performance
2. Run BootstrapFewShot optimization
3. Evaluate optimized performance
4. Save the optimized program

In [ ]:
import sys, os, importlib
from pathlib import Path


ROOT = Path("c:\\Users\\gioan\\Documents\\GitHub\\chatbutt")

# Verify the path is correct
if not (ROOT / ".env").exists():
    # Fallback: try to find it by searching up from cwd
    ROOT = Path.cwd()
    for _ in range(6):
        if (ROOT / "notebooks" / "03_optimize.ipynb").exists():
            break
        parent = ROOT.parent
        if parent == ROOT:
            break
        ROOT = parent

sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import dspy
lm = dspy.LM("groq/llama-3.3-70b-versatile", api_key=os.getenv("GROQ_API_KEY"))
dspy.configure(lm=lm)

print(f"Root: {ROOT}")
print(f"LM: {lm.model}")

Root: c:\Users\gioan\Documents\GitHub\chatbutt
LM: groq/llama-3.3-70b-versatile


## 1. Load Data & Initialize Chatbot

In [18]:
from src.modules import HiligaynonChatbot
from src.metrics import translation_relevance_metric, load_examples
from src.optimize import evaluate_program, optimize_bootstrap_fewshot, save_program

# Load train/dev sets
trainset = load_examples(ROOT / "data" / "examples" / "trainset.json")
devset = load_examples(ROOT / "data" / "examples" / "devset.json")
print(f"Train: {len(trainset)} examples, Dev: {len(devset)} examples")
print(f"Sample: {trainset[0]}")

# Initialize chatbot
chatbot = HiligaynonChatbot(chroma_dir=ROOT / "chroma_db")
print(f"Chatbot ready ({chatbot.collection.count()} entries in index)")

Train: 60 examples, Dev: 30 examples
Sample: Example({'hiligaynon': 'asawa', 'english': 'spouse: husband, wife; married (said of a woman)', 'definition': 'spouse: husband, wife; married (said of a woman)'}) (input_keys={'english', 'hiligaynon'})
Chatbot ready (10000 entries in index)


INFO:backoff:Backing off send_request(...) for 0.1s (requests.exceptions.ReadTimeout: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Read timed out. (read timeout=15))


## 2. Baseline Evaluation (Unoptimized)

In [20]:
import time

print("Running baseline evaluation on dev set...")
print("(This will evaluate on 30 examples with rate limiting)\n")

baseline_score = None
max_retries = 3

for attempt in range(max_retries):
    try:
        baseline_score = evaluate_program(chatbot, devset)
        print(f"\n🎯 Baseline score: {baseline_score:.2f}%")
        break
    except Exception as e:
        error_msg = str(e).lower()
        if "rate_limit" in error_msg or "429" in error_msg or "cancelled" in error_msg:
            print(f"\n⚠️ Attempt {attempt+1} failed: Rate limit or execution error")
            if attempt < max_retries - 1:
                wait_time = 10 * (attempt + 1)
                print(f"   Waiting {wait_time}s before retry...")
                time.sleep(wait_time)
            else:
                print("\n❌ Max retries reached. Options:")
                print("   1. Wait 1+ hours for daily limit reset (Groq free tier: 100K TPD)")
                print("   2. Upgrade to Groq Dev Tier: https://console.groq.com/settings/billing")
                print("   3. Run on smaller devset: evaluate_program(chatbot, devset[:10])")
                print("\nSkipping to optimization with unoptimized baseline...")
                baseline_score = 50.0  # Placeholder
        else:
            raise

Running baseline evaluation on dev set...
(This will evaluate on 30 examples with rate limiting)

  0%|          | 0/30 [00:00<?, ?it/s]

2026/02/12 11:21:48 ERROR dspy.utils.parallelizer: Error for Example({'hiligaynon': 'adlaw', 'english': 'day; sun', 'definition': 'day; sun'}) (input_keys={'english', 'hiligaynon'}): litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01hqkyd6tge7ztb4dsfyz787bd` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99850, Requested 1349. Please try again in 17m15.936s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
. Set `provide_traceback=True` for traceback.


Average Metric: 0.00 / 0 (0%):   3%|▎         | 1/30 [00:09<04:32,  9.39s/it]

2026/02/12 11:21:57 ERROR dspy.utils.parallelizer: Error for Example({'hiligaynon': 'aslom', 'english': 'make sour', 'definition': 'make sour'}) (input_keys={'english', 'hiligaynon'}): litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01hqkyd6tge7ztb4dsfyz787bd` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99838, Requested 1597. Please try again in 20m39.84s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
. Set `provide_traceback=True` for traceback.


Average Metric: 0.00 / 0 (0%):   7%|▋         | 2/30 [00:19<04:28,  9.58s/it]

...

Average Metric: 0.00 / 0 (0%):  30%|███       | 9/30 [01:28<03:07,  8.94s/it]

2026/02/12 11:27:08 ERROR dspy.utils.parallelizer: Error for Example({'hiligaynon': 'abo', 'english': 'ash(es); to make ashes', 'definition': 'ash(es); to make ashes'}) (input_keys={'english', 'hiligaynon'}): litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01hqkyd6tge7ztb4dsfyz787bd` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99480, Requested 2398. Please try again in 27m2.592s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
. Set `provide_traceback=True` for traceback.


Average Metric: 0.00 / 0 (0%): 100%|██████████| 30/30 [01:36<00:00,  3.22s/it]

2026/02/12 11:27:08 WARNING dspy.utils.parallelizer: Execution cancelled due to errors or interruption.




⚠️ Attempt 3 failed: Rate limit or execution error

❌ Max retries reached. Options:
   1. Wait 1+ hours for daily limit reset (Groq free tier: 100K TPD)
   2. Upgrade to Groq Dev Tier: https://console.groq.com/settings/billing
   3. Run on smaller devset: evaluate_program(chatbot, devset[:10])

Skipping to optimization with unoptimized baseline...


## 3. BootstrapFewShot Optimization

In [21]:
import time

print("Running BootstrapFewShot optimization...")
print("NOTE: This uses many LLM calls. If you hit Groq's daily limit (100K TPD),")
print("      wait for the limit to reset or switch to a paid tier.\n")

try:
    optimized = optimize_bootstrap_fewshot(
        chatbot,
        trainset,
        metric=translation_relevance_metric,
        max_bootstrapped_demos=4,
        max_labeled_demos=8,
    )
    print("✅ Optimization complete!")
except Exception as e:
    if "rate_limit" in str(e).lower() or "429" in str(e):
        print(f"\n⚠️ Rate limit hit: {str(e)[:200]}")
        print("\nOptions:")
        print("  1. Wait for daily limit reset and re-run this cell")
        print("  2. Upgrade Groq to Dev Tier (https://console.groq.com/settings/billing)")
        print("  3. Use a smaller trainset: optimize_bootstrap_fewshot(chatbot, trainset[:15], ...)")
        print("\nSaving unoptimized chatbot as fallback...")
        optimized = chatbot
    else:
        raise

Running BootstrapFewShot optimization...
NOTE: This uses many LLM calls. If you hit Groq's daily limit (100K TPD),
      wait for the limit to reset or switch to a paid tier.



  0%|          | 0/60 [00:00<?, ?it/s]2026/02/12 11:27:28 ERROR dspy.teleprompt.bootstrap: Failed to run or to evaluate example Example({'hiligaynon': 'asawa', 'english': 'spouse: husband, wife; married (said of a woman)', 'definition': 'spouse: husband, wife; married (said of a woman)'}) (input_keys={'english', 'hiligaynon'}) with <function translation_relevance_metric at 0x000001F47F611760> due to litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01hqkyd6tge7ztb4dsfyz787bd` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99456, Requested 1383. Please try again in 12m4.896s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
.
  2%|▏         | 1/60 [00:08<07:58,  8.12s/it]2026/02/12 11:27:36 ERROR dspy.teleprompt.bootstrap: Failed to run or to evaluate example Example({'hiligayn


⚠️ Rate limit hit: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01hqkyd6tge7ztb4dsfyz787bd` service tier `on_de

Options:
  1. Wait for daily limit reset and re-run this cell
  2. Upgrade Groq to Dev Tier (https://console.groq.com/settings/billing)
  3. Use a smaller trainset: optimize_bootstrap_fewshot(chatbot, trainset[:15], ...)

Saving unoptimized chatbot as fallback...


## 4. Evaluate Optimized Program

In [22]:
print("Evaluating optimized program on dev set...")
optimized_score = evaluate_program(optimized, devset)

print(f"\n📊 Results:")
print(f"   Baseline:  {baseline_score:.2f}%")
print(f"   Optimized: {optimized_score:.2f}%")
print(f"   Improvement: {optimized_score - baseline_score:+.2f}%")

Evaluating optimized program on dev set...
  0%|          | 0/30 [00:00<?, ?it/s]

2026/02/12 11:28:56 ERROR dspy.utils.parallelizer: Error for Example({'hiligaynon': 'adlaw', 'english': 'day; sun', 'definition': 'day; sun'}) (input_keys={'english', 'hiligaynon'}): litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01hqkyd6tge7ztb4dsfyz787bd` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99981, Requested 1364. Please try again in 19m22.08s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
. Set `provide_traceback=True` for traceback.


Average Metric: 0.00 / 0 (0%):   3%|▎         | 1/30 [00:08<03:56,  8.15s/it]

2026/02/12 11:29:05 ERROR dspy.utils.parallelizer: Error for Example({'hiligaynon': 'aslom', 'english': 'make sour', 'definition': 'make sour'}) (input_keys={'english', 'hiligaynon'}): litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01hqkyd6tge7ztb4dsfyz787bd` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99971, Requested 1611. Please try again in 22m46.848s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
. Set `provide_traceback=True` for traceback.


Average Metric: 0.00 / 0 (0%):   7%|▋         | 2/30 [00:16<03:48,  8.17s/it]

...

Average Metric: 0.00 / 0 (0%):  30%|███       | 9/30 [01:14<02:49,  8.07s/it]

2026/02/12 11:30:10 ERROR dspy.utils.parallelizer: Error for Example({'hiligaynon': 'abo', 'english': 'ash(es); to make ashes', 'definition': 'ash(es); to make ashes'}) (input_keys={'english', 'hiligaynon'}): litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01hqkyd6tge7ztb4dsfyz787bd` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99895, Requested 2398. Please try again in 33m1.151999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
. Set `provide_traceback=True` for traceback.


Average Metric: 0.00 / 0 (0%): 100%|██████████| 30/30 [01:22<00:00,  2.74s/it]

2026/02/12 11:30:10 WARNING dspy.utils.parallelizer: Execution cancelled due to errors or interruption.


Exception: Execution cancelled due to errors or interruption.

## 5. Save Optimized Program

In [23]:
save_path = ROOT / "optimized" / "hiligaynon_v1.json"
save_program(optimized, save_path)
print(f"\n✅ Saved to {save_path}")
print(f"   This program can be loaded by the Gradio app at app/gradio_app.py")

Saved optimized program to c:\Users\gioan\Documents\GitHub\chatbutt\optimized\hiligaynon_v1.json

✅ Saved to c:\Users\gioan\Documents\GitHub\chatbutt\optimized\hiligaynon_v1.json
   This program can be loaded by the Gradio app at app/gradio_app.py


## 6. Quick Test with Optimized Program

In [24]:
# Test the optimized program
test_msgs = [
    "How do you say 'I love you' in Hiligaynon?",
    "What does 'maayo' mean?",
    "Translate 'balay' to English",
]

for msg in test_msgs:
    print(f"\n{'='*60}")
    print(f"📨 {msg}")
    result = optimized(user_message=msg)
    print(f"💬 {result.response[:300]}")
    print(f"   [lang={result.input_language}, intent={result.intent}]")


📨 How do you say 'I love you' in Hiligaynon?


RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01hqkyd6tge7ztb4dsfyz787bd` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99884, Requested 1562. Please try again in 20m49.344s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
